# Week 4: Pipeline Integration, Unsupervised Pattern Mining & External Tool Visualization
**Course:** Big Data | **Project:** Urban Air Quality Predictive Modeling  
**Phase 7 — Production Readiness, Advanced Evaluation & Chapter 9 Integration**

---

## Overview

This notebook builds on previous weeks by formalizing the model into a production-ready pipeline **and** extending the analysis with unsupervised learning (FP-Growth association rule mining) and professional interactive visualization (Plotly). These additions directly address the Chapter 9 topics on unsupervised learning and external tool integration.

| # | Objective | Chapter 9 Tie-in | Owner |
|---|-----------|-----------------|-------|
| 1 | Wrap preprocessing + model into a single `Pipeline` object | Big Data pipeline integration | Ian |
| 2 | Validate model stability using 5-Fold Cross-Validation | Algorithm metrics | Polly & Erick |
| 3 | Evaluate performance with diagnostic visualizations (Plotly interactive) | Visualize processed data professionally | Daryl |
| 4 | **FP-Growth association rule mining on pollutant co-occurrence patterns** | Chapter 9: Unsupervised learning / FP-Growth | All |
| 5 | Serialize and export the pipeline; provide Streamlit app code | External tool integration | Danilyn |

> **Target variable:** `pm2_5` (fine particulate matter, µg/m³)  
> **Features used:** `co` (carbon monoxide), `no2` (nitrogen dioxide), `so2` (sulfur dioxide)

## Setup & Data Loading

We import all dependencies up front, then load and clean the dataset. New this week: **`plotly`** replaces static matplotlib for interactive charts, and **`mlxtend`** provides a pandas-compatible FP-Growth implementation (no Spark cluster required for the notebook; the PySpark equivalent is shown in Section 4).

The cleaning step drops rows with missing values in the model's required columns only, keeping the rest of the dataset intact for the association rule mining section.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

# Interactive visualization (replaces static matplotlib for professional output)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Supervised learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Unsupervised learning — FP-Growth (Chapter 9)
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

# ── Configuration ──────────────────────────────────────────────────────────────
DATA_PATH    = 'clean_air_quality_data.csv'
MODEL_PATH   = 'urban_air_quality_v1.joblib'
FEATURES     = ['co', 'no2', 'so2']
TARGET       = 'pm2_5'
NROWS        = 700_000
RANDOM_STATE = 42
TEST_SIZE    = 0.2

# ── Load ───────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, nrows=NROWS)
print(f"Loaded {len(df):,} rows from '{DATA_PATH}'")

# ── Clean: drop rows with missing values in relevant columns only ──────────────
cols_required = FEATURES + [TARGET]
n_before = len(df)
df = df.dropna(subset=cols_required)
n_dropped = n_before - len(df)
print(f"Dropped {n_dropped:,} rows with missing values ({n_dropped/n_before:.1%} of total)")

# ── Split ──────────────────────────────────────────────────────────────────────
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"\nData split complete:")
print(f"  Training set : {len(X_train):,} rows")
print(f"  Test set     : {len(X_test):,} rows")
print(f"\nFeature summary:")
display(X_train.describe().round(3))

Loaded 700,000 rows from 'clean_air_quality_data.csv'
Dropped 25,144 rows with missing values (3.6% of total)

Data split complete:
  Training set : 539,884 rows
  Test set     : 134,972 rows

Feature summary:


,co,no2,so2
count,539884.000,539884.000,539884.000
mean,0.313,10.026,1.229
std,0.202,7.815,0.995
min,0.000,0.000,0.000
25%,0.190,4.500,0.700
50%,0.270,8.000,1.100
75%,0.380,13.300,1.600
max,5.320,77.800,80.400


## 1. Pipeline Development
*By Ian (Pipeline Architecture)*

- **No data leakage** : the `StandardScaler` is fitted only on training data. When the pipeline is used for inference, it automatically applies the same scaling parameters learned during training.
- **Portability** : saving the pipeline to disk with `joblib` captures both the scaler *and* the model, so any downstream application (e.g., Streamlit) just loads one file and calls `.predict()`.

The `RandomForestRegressor` uses parameters carried forward from our Week 3 tuning (`max_depth=20`, `n_estimators=100`). We also extract **feature importances** here, these tell us which pollutant is the strongest predictor of PM2.5, which connects to the broader data exploration goal.

In [3]:

# ── Model hyperparameters (from Week 3 tuning) ─────────────────────────────────
rf_params = {
    'n_estimators'     : 100,
    'max_depth'        : 20,
    'min_samples_split': 5,
    'random_state'     : RANDOM_STATE,
    'n_jobs'           : -1,
}

# ── Build pipeline: StandardScaler → RandomForestRegressor ────────────────────
model_pipeline = Pipeline([
    ('scaler',    StandardScaler()),
    ('regressor', RandomForestRegressor(**rf_params)),
])

print("Pipeline structure:")
print(model_pipeline)

# ── Train ──────────────────────────────────────────────────────────────────────
print("\nFitting pipeline on training data (this may take a moment)...")
model_pipeline.fit(X_train, y_train)
print("Training complete.")

# ── Predict on held-out test set ───────────────────────────────────────────────
y_pred = model_pipeline.predict(X_test)

# ── Test-set evaluation metrics ────────────────────────────────────────────────
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n── Test Set Performance ──")
print(f"  MAE  : {mae:.4f} µg/m³")
print(f"  RMSE : {rmse:.4f} µg/m³")
print(f"  R²   : {r2:.4f}")

# ── Feature Importances ────────────────────────────────────────────────────────
importances = model_pipeline.named_steps['regressor'].feature_importances_
feat_df = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})\
            .sort_values('Importance', ascending=False)

print("\n── Feature Importances ──")
display(feat_df.round(4))

fig_imp = px.bar(
    feat_df, x='Feature', y='Importance',
    title='Random Forest Feature Importances — PM2.5 Predictors',
    color='Importance',
    color_continuous_scale='Oranges',
    labels={'Importance': 'Importance Score'},
    template='plotly_white'
)
fig_imp.update_layout(coloraxis_showscale=False)
fig_imp.show()

Pipeline structure:
Pipeline(steps=[('scaler', StandardScaler()),
                ('regressor',
                 RandomForestRegressor(max_depth=20, min_samples_split=5,
                                       n_jobs=-1, random_state=42))])

Fitting pipeline on training data (this may take a moment)...
Training complete.

── Test Set Performance ──
  MAE  : 6.2320 µg/m³
  RMSE : 8.4123 µg/m³
  R²   : 0.3622

── Feature Importances ──


,Feature,Importance
0,co,0.6103
1,no2,0.2019
2,so2,0.1878


## 2. Cross-Validation
*By Polly & Erick (Validation & Metrics)*

A single train/test split gives one estimate of model performance, which may be influenced by how the data was randomly divided. **5-Fold Cross-Validation** addresses this by training and evaluating the model on 5 different non-overlapping subsets of the training data, then averaging the results.

- A **high mean R²** confirms the model explains most of the variance in PM2.5 levels.
- A **low standard deviation** means the model performs consistently, not just well on one lucky split.

Note: Cross-validation is run on `X_train` only. The test set remains untouched to give an unbiased final evaluation.

In [4]:
print("Running 5-Fold Cross-Validation on training data...")

cv_scores = cross_val_score(
    model_pipeline, X_train, y_train, cv=5, scoring='r2', n_jobs=-1
)

print("\n── Cross-Validation Results (R²) ──")
for fold, score in enumerate(cv_scores, start=1):
    print(f"  Fold {fold}: {score:.4f}")
print(f"  {'─'*22}")
print(f"  Mean  : {cv_scores.mean():.4f}")
print(f"  Std   : {cv_scores.std():.4f}")

if cv_scores.std() < 0.01:
    print("\n→ Low variance across folds: model is stable and generalizes well.")
else:
    print("\n→ Some variance across folds detected — worth investigating data distribution.")

# ── Interactive CV bar chart ───────────────────────────────────────────────────
cv_df = pd.DataFrame({'Fold': [f'Fold {i}' for i in range(1, 6)], 'R²': cv_scores})
fig_cv = px.bar(
    cv_df, x='Fold', y='R²',
    title='5-Fold Cross-Validation R² Scores',
    color='R²',
    color_continuous_scale='Blues',
    range_y=[max(0, cv_scores.min() - 0.05), 1.0],
    template='plotly_white'
)
fig_cv.add_hline(
    y=cv_scores.mean(), line_dash='dash', line_color='red',
    annotation_text=f'Mean R² = {cv_scores.mean():.4f}',
    annotation_position='top left'
)
fig_cv.update_layout(coloraxis_showscale=False)
fig_cv.show()

Running 5-Fold Cross-Validation on training data...

── Cross-Validation Results (R²) ──
  Fold 1: 0.3541
  Fold 2: 0.3587
  Fold 3: 0.3595
  Fold 4: 0.3550
  Fold 5: 0.3541
  ──────────────────────
  Mean  : 0.3563
  Std   : 0.0023

→ Low variance across folds: model is stable and generalizes well.


## 3. Diagnostic Visualizations (Plotly — Interactive)
*By Daryl (Visualization & Evaluation)*

Standard accuracy metrics (MAE, R²) tell us *how much* error there is, but not *where* it comes from. Three diagnostic plots provide a deeper qualitative check:

1. **Residuals vs. Predicted Values** — Residuals should scatter randomly around zero with no pattern. Funnel shapes indicate heteroscedasticity.
2. **Error Distribution** — A bell-shaped distribution centered near zero suggests errors are random rather than systematic.
3. **Actual vs. Predicted** — Points close to the diagonal (perfect-prediction line) represent accurate predictions.

This week these are rendered with **Plotly** for interactive hover, zoom, and pan — meeting the *professional format* requirement of the lab.

In [ ]:
residuals = y_test.values - y_pred

# Downsample for rendering performance 
N_SAMPLE = min(10_000, len(y_test))
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(y_test), size=N_SAMPLE, replace=False)

fig_diag = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Residuals vs. Predicted Values',
        'Distribution of Prediction Errors',
        'Actual vs. Predicted PM2.5'
    ]
)

# Panel 1 — Residuals vs Predicted
fig_diag.add_trace(
    go.Scatter(
        x=y_pred[idx], y=residuals[idx],
        mode='markers',
        marker=dict(size=3, color='#E8651A', opacity=0.4),
        name='Residual',
        hovertemplate='Predicted: %{x:.2f}<br>Residual: %{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)
fig_diag.add_hline(y=0, line_dash='dash', line_color='black', row=1, col=1)

# Panel 2 — Error histogram
fig_diag.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=60,
        marker_color='#F5A461',
        name='Error',
        hovertemplate='Error bin: %{x:.2f}<br>Count: %{y}<extra></extra>'
    ),
    row=1, col=2
)
fig_diag.add_vline(x=0, line_dash='dash', line_color='black', row=1, col=2)

# Panel 3 — Actual vs Predicted
fig_diag.add_trace(
    go.Scatter(
        x=y_test.values[idx], y=y_pred[idx],
        mode='markers',
        marker=dict(size=3, color='#3A7EBF', opacity=0.35),
        name='Prediction',
        hovertemplate='Actual: %{x:.2f}<br>Predicted: %{y:.2f}<extra></extra>'
    ),
    row=1, col=3
)
lims = [float(min(y_test.min(), y_pred.min())), float(max(y_test.max(), y_pred.max()))]
fig_diag.add_trace(
    go.Scatter(
        x=lims, y=lims,
        mode='lines',
        line=dict(color='black', dash='dash', width=1),
        name='Perfect prediction'
    ),
    row=1, col=3
)

fig_diag.update_layout(
    title_text='Model Diagnostic Dashboard — Week 4',
    title_font_size=16,
    height=450,
    template='plotly_white',
    showlegend=False
)
fig_diag.update_xaxes(title_text='Predicted PM2.5 (µg/m³)', row=1, col=1)
fig_diag.update_yaxes(title_text='Residual (Actual − Predicted)', row=1, col=1)
fig_diag.update_xaxes(title_text='Error (Actual − Predicted)', row=1, col=2)
fig_diag.update_yaxes(title_text='Count', row=1, col=2)
fig_diag.update_xaxes(title_text='Actual PM2.5 (µg/m³)', row=1, col=3)
fig_diag.update_yaxes(title_text='Predicted PM2.5 (µg/m³)', row=1, col=3)

fig_diag.show()

## 4. Unsupervised Learning — FP-Growth Association Rule Mining
*By All (Chapter 9 Integration)*

### Why association rule mining here?

Our supervised model predicts *how much* PM2.5 there will be. But a complementary question is: **which pollutant levels tend to appear together?** If high CO, high NO2, and high PM2.5 almost always co-occur, that pattern has direct policy implications, targeting CO sources may reduce PM2.5 as a side effect.

This is exactly the kind of co-occurrence analysis covered in Chapter 9 (Association Rule Mining). We apply it to air quality data by:

1. **Discretizing** each continuous pollutant reading into three ordinal categories: `low`, `medium`, `high` (using tertile bins).
2. **Encoding** each row (station-timestamp observation) as a "transaction" of categorical items (e.g., `co_high`, `no2_medium`, `pm25_high`).
3. **Running FP-Growth** to find frequent itemsets, then generating association rules.
4. **Interpreting** rules by support, confidence, and lift, the three measures defined in Chapter 9.

### PySpark equivalent (for large-scale deployment)

The cell below uses `mlxtend` (pandas-native, no cluster needed). The equivalent Spark code using `pyspark.ml.fpm.FPGrowth` is provided as a commented block at the end of the section for reference.

In [ ]:
# ── Step 1: Discretize continuous readings into Low / Medium / High ────────────
# We use quantile-based (tertile) bins so each category has roughly equal
# frequency — this avoids class imbalance in the transaction encoding.

aq_cols = FEATURES + [TARGET]   # ['co', 'no2', 'so2', 'pm2_5']
label_map = {0: 'low', 1: 'medium', 2: 'high'}

df_discrete = df[aq_cols].copy()
for col in aq_cols:
    df_discrete[col] = pd.qcut(df[col], q=3, labels=False, duplicates='drop')
    df_discrete[col] = df_discrete[col].map(label_map)

print("Discretized value counts (sample — co column):")
print(df_discrete['co'].value_counts())
display(df_discrete.head(5))

Discretized value counts (sample — co column):
co
low       241545
medium    218622
high      214689
Name: count, dtype: int64


,co,no2,so2,pm2_5
0,low,low,medium,medium
1,medium,medium,high,medium
2,low,low,low,medium
3,low,low,low,medium
4,low,low,medium,medium


In [7]:
# ── Step 2: Build transactions ─────────────────────────────────────────────────
# Each row becomes a set of labelled items: e.g. {'co_high', 'no2_medium', 'pm25_low'}
# Column names are renamed to match Chapter 9 notation (antecedent / consequent items).

col_rename = {'co': 'co', 'no2': 'no2', 'so2': 'so2', 'pm2_5': 'pm25'}

transactions = []
for _, row in df_discrete.iterrows():
    items = [f"{col_rename[col]}_{row[col]}" for col in aq_cols if pd.notna(row[col])]
    transactions.append(items)

# Use a sample for speed (FP-Growth on 700k rows may be slow without Spark)
SAMPLE_N = 50_000
rng2 = np.random.default_rng(RANDOM_STATE)
transactions_sample = rng2.choice(transactions, size=min(SAMPLE_N, len(transactions)), replace=False).tolist()

print(f"Total transactions: {len(transactions):,}")
print(f"Using sample of  : {len(transactions_sample):,} for FP-Growth")
print(f"\nExample transaction: {transactions_sample[0]}")

# ── Step 3: One-hot encode for mlxtend ────────────────────────────────────────
te = TransactionEncoder()
te_array = te.fit_transform(transactions_sample)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)
print(f"\nEncoded shape: {df_encoded.shape}  (rows × unique items)")
print("Items:", list(df_encoded.columns))

Total transactions: 674,856
Using sample of  : 50,000 for FP-Growth

Example transaction: ['co_medium', 'no2_low', 'so2_high', 'pm25_high']

Encoded shape: (50000, 12)  (rows × unique items)
Items: ['co_high', 'co_low', 'co_medium', 'no2_high', 'no2_low', 'no2_medium', 'pm25_high', 'pm25_low', 'pm25_medium', 'so2_high', 'so2_low', 'so2_medium']


In [8]:
# ── Step 4: Run FP-Growth + generate association rules ────────────────────────
# minSupport=0.20 — itemset must appear in ≥20% of observations to be frequent.
# minConfidence=0.60 — rule must be correct ≥60% of the time it fires.
# These thresholds mirror the Chapter 9 examples.

MIN_SUPPORT    = 0.20
MIN_CONFIDENCE = 0.60

frequent_itemsets = fpgrowth(df_encoded, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print(f"Frequent itemsets found: {len(frequent_itemsets)}")
display(
    frequent_itemsets.sort_values('support', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

# ── Generate rules ─────────────────────────────────────────────────────────────
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules = rules.sort_values('lift', ascending=False)

print(f"\nAssociation rules found: {len(rules)}")

# Pretty-print the top rules
rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules_display = rules_display.rename(columns={
    'antecedents': 'Antecedent (X)',
    'consequents': 'Consequent (Y)',
    'support': 'Support',
    'confidence': 'Confidence',
    'lift': 'Lift'
})
display(rules_display.round(4).head(20).reset_index(drop=True))

Frequent itemsets found: 15


,support,itemsets,length
0,0.36098,frozenset({so2_low}),1
1,0.35796,frozenset({co_low}),1
2,0.34844,frozenset({pm25_low}),1
3,0.34762,frozenset({pm25_medium}),1
4,0.34360,frozenset({so2_medium}),1
5,0.34016,frozenset({no2_low}),1
6,0.33018,frozenset({no2_medium}),1
7,0.32966,frozenset({no2_high}),1
8,0.32120,frozenset({co_high}),1
9,0.32084,frozenset({co_medium}),1



Association rules found: 6


,Antecedent (X),Consequent (Y),Support,Confidence,Lift
0,co_high,no2_high,0.2439,0.7594,2.3036
1,no2_high,co_high,0.2439,0.7399,2.3036
2,no2_low,co_low,0.2523,0.7418,2.0722
3,co_low,no2_low,0.2523,0.7049,2.0722
4,pm25_low,co_low,0.2230,0.6400,1.7879
5,co_low,pm25_low,0.2230,0.6230,1.7879


In [ ]:
# ── Step 5: Visualize association rules — Support vs. Confidence, colored by Lift
# Lift > 1.0 means the rule is stronger than random chance.

rules_plot = rules_display.copy()
rules_plot['Rule'] = rules_plot['Antecedent (X)'] + '  →  ' + rules_plot['Consequent (Y)']

fig_rules = px.scatter(
    rules_plot,
    x='Support', y='Confidence',
    color='Lift',
    size='Lift',
    hover_name='Rule',
    color_continuous_scale='RdYlGn',
    title='FP-Growth Association Rules — Support vs. Confidence (color = Lift)',
    labels={'Support': 'Support', 'Confidence': 'Confidence', 'Lift': 'Lift'},
    template='plotly_white'
)
fig_rules.add_hline(y=MIN_CONFIDENCE, line_dash='dot', line_color='grey',
                    annotation_text=f'Min confidence = {MIN_CONFIDENCE}')
fig_rules.add_vline(x=MIN_SUPPORT, line_dash='dot', line_color='grey',
                    annotation_text=f'Min support = {MIN_SUPPORT}')
fig_rules.show()

# ── Top rules by Lift (bar chart) ──────────────────────────────────────────────
top_rules = rules_plot.nlargest(10, 'Lift')
fig_top = px.bar(
    top_rules, x='Lift', y='Rule', orientation='h',
    color='Confidence',
    color_continuous_scale='Blues',
    title='Top 10 Association Rules by Lift',
    template='plotly_white',
    height=420
)
fig_top.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_top.show()

In [14]:
# Export for Tableau
rules_display.to_csv('association_rules_export.csv', index=False)
print("Saved association_rules_export.csv — we will load this into Tableau")

Saved association_rules_export.csv — we will load this into Tableau


### PySpark FP-Growth (Big Data / Distributed Equivalent)

The `mlxtend` implementation above runs locally. For large-scale deployment on a Spark cluster, the equivalent code using `pyspark.ml.fpm.FPGrowth` is shown below. The input format, parameters (`minSupport`, `minConfidence`), and outputs (`freqItemsets`, `associationRules`) map directly to Chapter 9's Spark example.

```python
# ── PySpark FP-Growth equivalent ──────────────────────────────────────────────
# Requires: a running SparkSession and the transactions list built above.

from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth

spark = SparkSession.builder.appName("AirQualityAssociationMining").getOrCreate()

# Convert the same transactions list to a Spark DataFrame
spark_df = spark.createDataFrame(
    [(items,) for items in transactions_sample],
    ["items"]
)

fp = FPGrowth(
    itemsCol="items",
    minSupport=0.20,
    minConfidence=0.60
)

model = fp.fit(spark_df)

print("Frequent Itemsets:")
model.freqItemsets.show(truncate=False)

print("Association Rules:")
model.associationRules.show(truncate=False)

# Filter for lift > 1.0 (only rules stronger than random chance)
model.associationRules.filter("lift > 1.0").show(truncate=False)
```

## 5. Pipeline Export & Streamlit Integration
*By Danilyn (Deployment)*

The final step is serializing the complete pipeline (scaler + model) to a `.joblib` file. Because we saved the full `Pipeline` object — not just the `RandomForestRegressor` — the Streamlit app only needs to call `.predict(raw_input)`. The scaler transformation is applied automatically.

The Streamlit app code is included below and can be saved as `app.py` in the same directory as the `.joblib` file. Running `streamlit run app.py` launches the interactive dashboard.

In [13]:
# ── Serialize the full pipeline (scaler + model) to disk ──────────────────────
joblib.dump(model_pipeline, MODEL_PATH)

file_size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f"✅ Pipeline saved: '{MODEL_PATH}' ({file_size_kb:.1f} KB)")
print(f"   Location: {os.path.abspath(MODEL_PATH)}")
print()
print("Pipeline is ready for integration into the Streamlit dashboard.")

# ── Write Streamlit app.py ─────────────────────────────────────────────────────
streamlit_code = '''
# app.py — Urban Air Quality PM2.5 Predictor
# Run with: streamlit run app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.graph_objects as go

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Urban Air Quality — PM2.5 Predictor",
    page_icon="🌫️",
    layout="wide"
)

# ── Load model ────────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    return joblib.load("urban_air_quality_v1.joblib")

pipeline = load_model()

# ── Header ────────────────────────────────────────────────────────────────────
st.title("🌫️ Urban Air Quality — PM2.5 Predictor")
st.markdown(
    "Predict fine particulate matter (PM2.5 µg/m³) from gas pollutant readings. "
    "Uses a Random Forest model trained on the urban air quality dataset."
)

# ── Sidebar inputs ────────────────────────────────────────────────────────────
st.sidebar.header("Pollutant Readings")
co  = st.sidebar.number_input("CO — Carbon Monoxide",  min_value=0.0, value=1.0, step=0.1)
no2 = st.sidebar.number_input("NO2 — Nitrogen Dioxide", min_value=0.0, value=20.0, step=1.0)
so2 = st.sidebar.number_input("SO2 — Sulfur Dioxide",   min_value=0.0, value=5.0, step=0.5)

# ── Predict ───────────────────────────────────────────────────────────────────
input_df = pd.DataFrame([[co, no2, so2]], columns=["co", "no2", "so2"])
pm25_pred = pipeline.predict(input_df)[0]

# ── WHO thresholds ────────────────────────────────────────────────────────────
if pm25_pred <= 15:
    level, color = "Good", "green"
elif pm25_pred <= 35:
    level, color = "Moderate", "orange"
else:
    level, color = "Unhealthy", "red"

# ── Display result ────────────────────────────────────────────────────────────
col1, col2 = st.columns(2)
with col1:
    st.metric(label="Predicted PM2.5", value=f"{pm25_pred:.2f} µg/m³")
    st.markdown(f"**Air Quality Level:** :{color}[{level}]")

with col2:
    # Gauge chart
    fig_gauge = go.Figure(go.Indicator(
        mode="gauge+number",
        value=pm25_pred,
        title={"text": "PM2.5 µg/m³"},
        gauge={
            "axis": {"range": [0, 100]},
            "bar": {"color": color},
            "steps": [
                {"range": [0, 15], "color": "#d4edda"},
                {"range": [15, 35], "color": "#fff3cd"},
                {"range": [35, 100], "color": "#f8d7da"},
            ]
        }
    ))
    fig_gauge.update_layout(height=250, margin=dict(t=30, b=0, l=20, r=20))
    st.plotly_chart(fig_gauge, use_container_width=True)

# ── Sensitivity analysis ──────────────────────────────────────────────────────
st.subheader("Sensitivity Analysis — How PM2.5 changes with CO")
co_range = np.linspace(0, co * 3 + 1, 100)
sensitivity_df = pd.DataFrame({
    "co": co_range,
    "no2": no2,
    "so2": so2
})
sensitivity_df["pm25_pred"] = pipeline.predict(sensitivity_df)

fig_sens = px.line(
    sensitivity_df, x="co", y="pm25_pred",
    labels={"co": "CO level", "pm25_pred": "Predicted PM2.5 (µg/m³)"},
    title="Predicted PM2.5 vs. CO (holding NO2 and SO2 constant)",
    template="plotly_white"
)
fig_sens.add_vline(x=co, line_dash="dash", annotation_text="Current input")
st.plotly_chart(fig_sens, use_container_width=True)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code.strip())

print("✅ Streamlit app saved to 'app.py'")
print("   To launch: streamlit run app.py")

✅ Pipeline saved: 'urban_air_quality_v1.joblib' (744569.9 KB)
   Location: c:\Users\USER\Downloads\urban_air_quality_v1.joblib

Pipeline is ready for integration into the Streamlit dashboard.
✅ Streamlit app saved to 'app.py'
   To launch: streamlit run app.py


## Summary

This week we transitioned from a raw model to a deployment-ready artifact and extended the analysis with unsupervised learning:

| Component | What was done | Chapter 9 connection |
|-----------|--------------|---------------------|
| **Pipeline** | Scaler + RF bundled into one object; prevents data leakage | Big Data integration |
| **Feature Importances** | Bar chart showing CO, NO2, SO2 relative predictive power | Algorithm metrics |
| **Cross-Validation** | 5-fold CV with interactive Plotly bar chart | Algorithm metrics |
| **Diagnostic Plots** | Interactive Plotly residuals / error dist / actual vs. predicted | Professional visualization |
| **FP-Growth (mlxtend)** | Discretized pollutants → transaction encoding → association rules | Chapter 9: Unsupervised / FP-Growth |
| **FP-Growth (PySpark)** | Commented equivalent for Spark cluster deployment | Chapter 9: Spark MLlib FPGrowth |
| **Streamlit app** | Full `app.py` with gauge, sensitivity analysis, interactive charts | External tool integration |

### Key FP-Growth findings (interpret from rules table above)
- Rules where `pm25_high` appears as the **consequent** and have **lift > 1** identify pollutant combinations that are disproportionately associated with dangerous PM2.5 levels.
- High-lift rules with high confidence are the most actionable: they suggest which co-occurring conditions most reliably predict a PM2.5 spike.
- Association does **not** imply causation — a strong rule only means items appear together frequently, not that one causes the other.
